# Valida??o das fontes ? previs?o de eletivas Curitiba

Notebook de diagn?stico **somente leitura** para conferir o contrato de dados antes de executar `run_experimentos`.

- L? as tabelas existentes com Spark; n?o grava tabelas, arquivos, jobs ou logs.
- Restringe cirurgias e agendas ao `hub = Curitiba` e ao estabelecimento 1, Hospital Nossa Senhora do Pilar S.A.; pacientes e calendário permanecem filtrados conforme suas chaves.
- Exibe apenas schemas e contagens agregadas; n?o exibe identificadores de pacientes ou cirurgias.
- Confere `nr_cirurgia`, os timestamps e o crit?rio de 24 horas sem converter hor?rios de fuso.

Execute em um cluster Databricks com acesso de leitura ?s tabelas listadas abaixo. Se uma tabela ou coluna n?o estiver acess?vel, o notebook mostrar? o erro para orientar o ajuste.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

HUB = "Curitiba"
ESTABLISHMENT_CODE = "1"
ESTABLISHMENT_NAME = "Hospital Nossa Senhora do Pilar S.A."
SURGERY_TABLE = "bi_dev.gold.hc_cirurgia"
AGENDA_TABLE = "bi_dev.gold.hc_agenda_cirurgica"
PEOPLE_TABLE = "bi_dev.gold.hc_pessoa_fisica"
CALENDAR_TABLE = "planejamento_integrado.gold.calendario_curitiba"

surgery_required = {
    "hub", "nr_cirurgia", "dt_inicio_real", "ds_status_cirurgia",
    "ds_procedimento", "cd_pessoa_fisica", "ds_especialidade_cirurgiao",
    "nr_atendimento", "cd_medico_cirurgiao", "ds_sala", "cd_estabelecimento",
}
agenda_required = {
    "hub", "nr_cirurgia", "dt_agenda", "dt_agendamento", "cd_medico",
    "cd_pessoa_fisica", "ds_agenda", "ds_procedimento", "ds_especialidade",
    "dt_cancelamento", "ds_motivo_cancelamento", "cd_motivo_cancelamento",
    "ds_status_agenda",
    "ds_estabelecimento",
}
people_required = {"hub", "cd_pessoa_fisica", "nr_anos"}
calendar_required = {"data", "feriado"}

print("Timezone da sess?o Spark (apenas informativo):", spark.conf.get("spark.sql.session.timeZone"))
print("Os timestamps ser?o comparados como est?o armazenados; este notebook n?o aplica convers?o de fuso.")

In [ ]:
def inspect_source(table_name, hub_column, hub_value, required_columns, establishment_column=None, establishment_value=None):
    frame = spark.table(table_name)
    if hub_column is not None:
        frame = frame.where(F.col(hub_column) == hub_value)
    if establishment_column == "cd_estabelecimento":
        frame = frame.where(F.trim(F.col(establishment_column).cast("string")) == establishment_value)
    elif establishment_column == "ds_estabelecimento":
        frame = frame.where(F.lower(F.trim(F.col(establishment_column))) == establishment_value.casefold())
    missing = sorted(required_columns - set(frame.columns))
    schema_rows = [
        (field.name, field.dataType.simpleString(), field.nullable)
        for field in frame.schema.fields
    ]
    print(f"Fonte: {table_name} | filtro: {hub_column} = {hub_value}")
    if establishment_column is not None:
        print(f"Estabelecimento: {establishment_column} = {establishment_value}")
    print("Colunas obrigat?rias ausentes:", missing if missing else "nenhuma")
    display(spark.createDataFrame(schema_rows, ["coluna", "tipo_spark", "nullable"]))
    return frame, missing

surgery_raw, surgery_missing = inspect_source(
    SURGERY_TABLE, "hub", HUB, surgery_required, "cd_estabelecimento", ESTABLISHMENT_CODE
)
agenda_raw, agenda_missing = inspect_source(
    AGENDA_TABLE, "hub", HUB, agenda_required, "ds_estabelecimento", ESTABLISHMENT_NAME
)
people_raw, people_missing = inspect_source(
    PEOPLE_TABLE, "hub", HUB, people_required
)
calendar_raw, calendar_missing = inspect_source(
    CALENDAR_TABLE, None, None, calendar_required
)

missing_by_table = {
    SURGERY_TABLE: surgery_missing,
    AGENDA_TABLE: agenda_missing,
    PEOPLE_TABLE: people_missing,
    CALENDAR_TABLE: calendar_missing,
}
if any(missing_by_table.values()):
    raise ValueError(f"H? colunas obrigat?rias ausentes: {missing_by_table}")

In [ ]:
def spark_type(frame, column):
    return frame.schema[column].dataType.simpleString()

# O c?digo da r?plica exige estes campos como timestamps com data e hora.
timestamp_types = {
    f"{SURGERY_TABLE}.dt_inicio_real": spark_type(surgery_raw, "dt_inicio_real"),
    f"{AGENDA_TABLE}.dt_agenda": spark_type(agenda_raw, "dt_agenda"),
    f"{AGENDA_TABLE}.dt_agendamento": spark_type(agenda_raw, "dt_agendamento"),
}
display(spark.createDataFrame(
    [(name, kind, kind in ("timestamp", "timestamp_ntz")) for name, kind in timestamp_types.items()],
    ["campo", "tipo_spark", "tipo_aceito_pela_implementacao"],
))
if any(kind not in ("timestamp", "timestamp_ntz") for kind in timestamp_types.values()):
    raise TypeError(f"Campo de data/hora sem tipo timestamp: {timestamp_types}")
if len(set(timestamp_types.values())) != 1:
    raise TypeError(f"Tipos de timestamp diferentes; a implementa??o n?o far? convers?o impl?cita: {timestamp_types}")

In [ ]:
# Resumo de volume, chaves e nulos. As sa?das n?o incluem os valores das chaves.
surgery_candidates = (
    surgery_raw
    .where(F.col("ds_status_cirurgia") == "Realizada")
    .where(F.col("dt_inicio_real").isNotNull())
    .where(F.col("nr_cirurgia").isNotNull())
    .where(~F.lower(F.col("ds_procedimento")).like("%rob?tica%"))
    .withColumn("nr_cirurgia_key", F.col("nr_cirurgia").cast("string"))
)
agenda_candidates = (
    agenda_raw
    .where(F.col("nr_cirurgia").isNotNull())
    .withColumn("nr_cirurgia_key", F.col("nr_cirurgia").cast("string"))
)

surgery_key_stats = surgery_candidates.groupBy("hub", "nr_cirurgia_key").agg(
    F.count(F.lit(1)).alias("source_rows"),
    F.min("dt_inicio_real").alias("dt_inicio_real"),
)
agenda_key_stats = agenda_candidates.groupBy("hub", "nr_cirurgia_key").agg(
    F.count(F.lit(1)).alias("agenda_rows"),
    F.min("dt_agendamento").alias("first_dt_agendamento"),
    F.sum(F.when(F.col("dt_agendamento").isNull(), 1).otherwise(0)).alias("null_creation_rows"),
)

linked = surgery_key_stats.join(agenda_key_stats, ["hub", "nr_cirurgia_key"], "left")
summary = linked.agg(
    F.count(F.lit(1)).alias("cirurgias_distintas_com_status_realizada_e_horario"),
    F.sum(F.when(F.col("agenda_rows").isNotNull(), 1).otherwise(0)).alias("cirurgias_com_agenda_correspondente"),
    F.sum(F.when(F.col("agenda_rows").isNull(), 1).otherwise(0)).alias("cirurgias_sem_agenda_correspondente"),
    F.sum(F.when(F.col("agenda_rows") > 1, 1).otherwise(0)).alias("cirurgias_com_mais_de_um_agendamento"),
    F.sum(F.when(F.col("source_rows") > 1, 1).otherwise(0)).alias("chaves_com_mais_de_uma_linha_na_fonte_cirurgia"),
    F.sum(F.when(F.col("first_dt_agendamento").isNull() & F.col("agenda_rows").isNotNull(), 1).otherwise(0)).alias("cirurgias_com_agenda_sem_dt_agendamento"),
    F.sum(F.when(
        F.col("first_dt_agendamento").isNotNull()
        & F.expr("dt_inicio_real >= first_dt_agendamento + INTERVAL 24 HOURS"), 1
    ).otherwise(0)).alias("eletivas_pela_regra_24h"),
    F.sum(F.when(
        F.col("first_dt_agendamento").isNotNull()
        & F.expr("dt_inicio_real < first_dt_agendamento + INTERVAL 24 HOURS"), 1
    ).otherwise(0)).alias("agendadas_com_menos_de_24h"),
)
print("Resumo do v?nculo por hub + nr_cirurgia, usando a primeira cria??o do agendamento:")
display(summary)

# Verifica empates na menor data/hora de cria??o. Qualquer linha empatada ? v?lida pela regra definida.
first_creation = agenda_key_stats.select("hub", "nr_cirurgia_key", "first_dt_agendamento")
tied_first = (
    agenda_candidates.join(first_creation, ["hub", "nr_cirurgia_key"], "inner")
    .where(F.col("dt_agendamento") == F.col("first_dt_agendamento"))
    .groupBy("hub", "nr_cirurgia_key")
    .agg(F.count(F.lit(1)).alias("first_rows"))
    .where(F.col("first_rows") > 1)
)
print("Quantidade de cirurgias com empate na primeira cria??o:", tied_first.count())

In [ ]:
# Confere o crit?rio para a agenda futura: hora prevista menos cria??o >= 24 horas.
future_agenda_candidates = agenda_candidates.where(
    F.col("dt_agenda").isNotNull() & F.col("dt_agendamento").isNotNull()
)
future_eligibility = future_agenda_candidates.agg(
    F.countDistinct(F.when(
        F.expr("dt_agenda >= dt_agendamento + INTERVAL 24 HOURS"),
        F.concat_ws("|", F.col("hub"), F.col("nr_cirurgia_key")),
    )).alias("cirurgias_com_alguma_agenda_prevista_elegivel"),
    F.countDistinct(F.when(
        F.expr("dt_agenda < dt_agendamento + INTERVAL 24 HOURS"),
        F.concat_ws("|", F.col("hub"), F.col("nr_cirurgia_key")),
    )).alias("cirurgias_com_agenda_prevista_abaixo_de_24h"),
)
display(future_eligibility)

# Cobertura b?sica do calend?rio da cidade usada pela previs?o di?ria.
calendar_summary = calendar_raw.agg(
    F.count(F.lit(1)).alias("linhas_calendario_curitiba"),
    F.min(F.to_date("data")).alias("primeira_data"),
    F.max(F.to_date("data")).alias("ultima_data"),
    F.sum(F.when(F.col("data").isNull(), 1).otherwise(0)).alias("linhas_sem_data"),
)
display(calendar_summary)
print("Valida??o conclu?da. Nenhuma escrita foi executada.")